# 04 - Causal Inference (A/B + DiD)

This notebook runs both randomized and quasi-experimental analyses.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

CAUSAL_DIR = ROOT / 'ml' / 'data' / 'reports' / 'causal'
AB_DIR = CAUSAL_DIR / 'ab_test'
DID_DIR = CAUSAL_DIR / 'did'
AB_DIR.mkdir(parents=True, exist_ok=True)
DID_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Cell 1: A/B data load + balance checks
ab_path = ROOT / 'ml' / 'data' / 'experiments' / 'ab_checkout_template.csv'
ab = pd.read_csv(ab_path)
ab['converted'] = ab['converted'].astype(int)
ab['revenue_30d'] = pd.to_numeric(ab['revenue_30d'], errors='coerce').fillna(0.0)
ab['pre_revenue_30d'] = pd.to_numeric(ab['pre_revenue_30d'], errors='coerce').fillna(0.0)

display(ab.head())
print('Variant counts:')
print(ab['variant'].value_counts())


In [ ]:
# Cell 2: A/B binary outcome (z-test)
g = ab.groupby('variant')['converted'].agg(['sum', 'count', 'mean']).rename(columns={'sum': 'successes', 'count': 'n', 'mean': 'rate'})
variants = sorted(g.index.tolist())
control, treatment = variants[0], variants[1]

count = np.array([g.loc[treatment, 'successes'], g.loc[control, 'successes']])
nobs = np.array([g.loc[treatment, 'n'], g.loc[control, 'n']])
z_stat, p_value = proportions_ztest(count, nobs)
rate_diff = float(g.loc[treatment, 'rate'] - g.loc[control, 'rate'])

print('control:', control, 'treatment:', treatment)
print('rate diff:', rate_diff)
print('z-stat:', z_stat, 'p-value:', p_value)


In [ ]:
# Cell 3: A/B continuous outcomes (Welch + CUPED)
control_vals = ab.loc[ab['variant'] == control, 'revenue_30d'].to_numpy(dtype=float)
treat_vals = ab.loc[ab['variant'] == treatment, 'revenue_30d'].to_numpy(dtype=float)

welch_t, welch_p = stats.ttest_ind(treat_vals, control_vals, equal_var=False)
mu_diff = float(np.mean(treat_vals) - np.mean(control_vals))

# CUPED
x = ab['pre_revenue_30d'].to_numpy(dtype=float)
y = ab['revenue_30d'].to_numpy(dtype=float)
theta = 0.0 if np.var(x, ddof=1) <= 0 else float(np.cov(y, x, ddof=1)[0,1] / np.var(x, ddof=1))
y_adj = y - theta * (x - x.mean())
ab['cuped_adjusted'] = y_adj

control_adj = ab.loc[ab['variant'] == control, 'cuped_adjusted'].to_numpy(dtype=float)
treat_adj = ab.loc[ab['variant'] == treatment, 'cuped_adjusted'].to_numpy(dtype=float)
cuped_t, cuped_p = stats.ttest_ind(treat_adj, control_adj, equal_var=False)
cuped_diff = float(np.mean(treat_adj) - np.mean(control_adj))

print('Welch revenue diff:', mu_diff, 'p:', welch_p)
print('CUPED diff:', cuped_diff, 'p:', cuped_p, 'theta:', theta)


In [ ]:
# Cell 4: Save A/B report
ab_results = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'control_label': control,
    'treatment_label': treatment,
    'binary_results': {
        'control_rate': float(g.loc[control, 'rate']),
        'treatment_rate': float(g.loc[treatment, 'rate']),
        'rate_diff': rate_diff,
        'p_value': float(p_value),
    },
    'continuous_results': {
        'metric': 'revenue_30d',
        'difference': mu_diff,
        'welch_p_value': float(welch_p),
    },
    'cuped_results': {
        'difference': cuped_diff,
        'welch_p_value': float(cuped_p),
        'theta': theta,
    },
}
(AB_DIR / 'ab_test_results.json').write_text(json.dumps(ab_results, indent=2), encoding='utf-8')
print('Saved A/B results:', AB_DIR / 'ab_test_results.json')


In [ ]:
# Cell 5: DiD data load
did_path = ROOT / 'ml' / 'data' / 'experiments' / 'did_panel_template.csv'
did = pd.read_csv(did_path)
did['outcome'] = pd.to_numeric(did['outcome'], errors='coerce')
did['treated'] = did['treated'].astype(int)
did['post'] = did['post'].astype(int)
did['period'] = pd.to_datetime(did['period'], errors='coerce')
did['treated_post'] = did['treated'] * did['post']

display(did.head())


In [ ]:
# Cell 6: Manual DiD
means = did.groupby(['treated', 'post'], as_index=False)['outcome'].mean()
display(means)

lookup = {(int(r.treated), int(r.post)): float(r.outcome) for _, r in means.iterrows()}
control_pre = lookup.get((0, 0), np.nan)
control_post = lookup.get((0, 1), np.nan)
treat_pre = lookup.get((1, 0), np.nan)
treat_post = lookup.get((1, 1), np.nan)

manual_did = (treat_post - treat_pre) - (control_post - control_pre)
print('Manual DiD:', manual_did)


In [ ]:
# Cell 7: Regression DiD + parallel trend test
did['period_str'] = did['period'].dt.strftime('%Y-%m')

model = smf.ols('outcome ~ treated + post + treated:post', data=did).fit(cov_type='HC3')
coef = float(model.params.get('treated:post', np.nan))
pval = float(model.pvalues.get('treated:post', np.nan))
print('Regression DiD coef:', coef, 'p-value:', pval)

pre = did[did['post'] == 0].copy()
parallel_p = None
if len(pre['period_str'].unique()) >= 2:
    order = {k: i for i, k in enumerate(sorted(pre['period_str'].unique().tolist()))}
    pre['t_num'] = pre['period_str'].map(order).astype(float)
    pre_model = smf.ols('outcome ~ treated + t_num + treated:t_num', data=pre).fit(cov_type='HC3')
    parallel_p = float(pre_model.pvalues.get('treated:t_num', np.nan))

print('Parallel trend p-value:', parallel_p)


In [ ]:
# Cell 8: Save DiD report + gate decision
alpha = 0.05
ab_ship = (ab_results['binary_results']['p_value'] < alpha and ab_results['binary_results']['rate_diff'] >= 0.0) or (ab_results['cuped_results']['welch_p_value'] < alpha)
did_ship = (pval < alpha and coef >= 0.0 and (parallel_p is None or parallel_p >= alpha))

final_decision = 'ship' if (ab_ship or did_ship) else 'iterate'

did_results = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'manual': {
        'control_pre': control_pre,
        'control_post': control_post,
        'treatment_pre': treat_pre,
        'treatment_post': treat_post,
        'did_manual': manual_did,
    },
    'regression': {
        'coef_treated_post': coef,
        'p_value': pval,
    },
    'parallel_trends_test': {
        'p_value': parallel_p,
    },
}
(DID_DIR / 'did_results.json').write_text(json.dumps(did_results, indent=2), encoding='utf-8')

decision = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'ab_recommend_ship': bool(ab_ship),
    'did_recommend_ship': bool(did_ship),
    'final_decision': final_decision,
}
(CAUSAL_DIR / 'causal_release_decision.json').write_text(json.dumps(decision, indent=2), encoding='utf-8')
print('Saved DiD results:', DID_DIR / 'did_results.json')
print('Saved causal decision:', CAUSAL_DIR / 'causal_release_decision.json')
print('Final decision:', final_decision)


## Next Notebook

Proceed to `05_cohort_time_series.ipynb`.
